# Kotoba STT - Real-time Speech-to-Text with Bidirectional Streaming

This notebook demonstrates how to use the Kotoba STT (Speech-to-Text) model package from AWS Marketplace.

**Product**: Kotoba STT - Real-time Speech Transcription  
**Seller**: Kotoba AI

## Key Features

- Real-time streaming transcription with sub-second latency
- Japanese and English language support
- Multiple audio formats (PCM16, Float32)
- Scalable GPU-accelerated inference

---

## IMPORTANT: Bidirectional Streaming Only

This product uses **SageMaker bidirectional streaming** exclusively:

- Standard `POST /invocations` is **NOT supported**
- Batch transform jobs are **NOT supported**
- Requires **Python 3.12+** with `aws-sdk-python[sagemaker-runtime-http2]`

---

## Prerequisites

1. **AWS Account** with SageMaker permissions
2. **Python 3.12+** (required for HTTP/2 SDK)
3. **AWS Marketplace subscription** to Kotoba STT

### Required IAM Permissions

```json
{
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "sagemaker:CreateEndpoint",
                "sagemaker:CreateEndpointConfig",
                "sagemaker:DeleteEndpoint",
                "sagemaker:DeleteEndpointConfig",
                "sagemaker:DescribeEndpoint",
                "sagemaker:InvokeEndpoint",
                "sagemaker:InvokeEndpointWithResponseStream"
            ],
            "Resource": "*"
        }
    ]
}
```

## Table of Contents

1. [Subscribe to Model Package](#1.-Subscribe-to-Model-Package)
2. [Setup](#2.-Setup)
3. [Deploy Real-time Endpoint](#3.-Deploy-Real-time-Endpoint)
4. [Perform Real-time Inference](#4.-Perform-Real-time-Inference)
5. [Clean-up](#5.-Clean-up)

## 1. Subscribe to Model Package

To subscribe to the Kotoba STT model package:

1. Open the [Kotoba STT listing on AWS Marketplace](#) <!-- TODO: Update with actual URL -->
2. Click **Continue to Subscribe**
3. Review the pricing and EULA, then click **Subscribe**
4. Once subscribed, click **Continue to Configuration**
5. Select your preferred region and note the **Model Package ARN**

## 2. Setup

### 2.1 Check Python Version

In [ ]:
import sys

if sys.version_info < (3, 12):
    raise RuntimeError(
        f"Python 3.12+ is required for SageMaker bidirectional streaming. "
        f"Current version: {sys.version}"
    )

print(f"Python version: {sys.version}")

### 2.2 Install Dependencies

In [ ]:
!pip install -q boto3 sagemaker numpy
!pip install -q "aws-sdk-python[sagemaker-runtime-http2]"

### 2.3 Import Libraries

In [ ]:
import asyncio
import base64
import json
import time
from io import BytesIO

import boto3
import numpy as np
import sagemaker
from sagemaker import ModelPackage

# SageMaker HTTP/2 SDK for bidirectional streaming
from aws_sdk_sagemaker_runtime_http2.client import SageMakerRuntimeHTTP2Client
from aws_sdk_sagemaker_runtime_http2.config import Config, HTTPAuthSchemeResolver
from aws_sdk_sagemaker_runtime_http2.models import (
    InvokeEndpointWithBidirectionalStreamInput,
    RequestPayloadPart,
    RequestStreamEventPayloadPart,
)
from smithy_aws_core.auth.sigv4 import SigV4AuthScheme
from smithy_aws_core.identity import StaticCredentialsResolver

print("All dependencies imported successfully.")

### 2.4 Configure AWS Session

In [ ]:
# Initialize SageMaker session
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sagemaker_session.boto_region_name

print(f"Region: {region}")
print(f"Role: {role}")

### 2.5 Configure Model Package ARN

Replace `<MODEL_PACKAGE_ARN>` with the ARN from your AWS Marketplace subscription.

In [ ]:
# TODO: Replace with your Model Package ARN from AWS Marketplace
MODEL_PACKAGE_ARN = "<MODEL_PACKAGE_ARN>"

# Validate ARN format
if MODEL_PACKAGE_ARN.startswith("<"):
    raise ValueError(
        "Please replace MODEL_PACKAGE_ARN with your actual Model Package ARN "
        "from AWS Marketplace subscription."
    )

## 3. Deploy Real-time Endpoint

### 3.1 Create Endpoint

In [ ]:
# Endpoint configuration
ENDPOINT_NAME = f"kotoba-stt-{int(time.time())}"
INSTANCE_TYPE = "ml.g6.2xlarge"  # GPU instance for real-time inference

print(f"Endpoint name: {ENDPOINT_NAME}")
print(f"Instance type: {INSTANCE_TYPE}")

In [ ]:
# Create Model from Model Package
model = ModelPackage(
    role=role,
    model_package_arn=MODEL_PACKAGE_ARN,
    sagemaker_session=sagemaker_session,
)

# Deploy endpoint
print(f"Deploying endpoint '{ENDPOINT_NAME}'...")
print("This may take 5-10 minutes.")

predictor = model.deploy(
    initial_instance_count=1,
    instance_type=INSTANCE_TYPE,
    endpoint_name=ENDPOINT_NAME,
)

print(f"Endpoint '{ENDPOINT_NAME}' is now InService.")

## 4. Perform Real-time Inference

### 4.1 Connection Protocol

Kotoba STT uses SageMaker bidirectional streaming:

```
Client (HTTP/2 SDK) → SageMaker Runtime (HTTPS:8443) → Sidecar → Container (WS:8080)
```

**Note**: Standard WebSocket libraries cannot connect directly to SageMaker endpoints.

### 4.2 Audio Format Specifications

| Format | Sample Rate | Channels | Encoding |
|--------|-------------|----------|----------|
| `pcm16` | 24000 Hz | 1 (mono) | Little-endian int16 |
| `float32` | 24000 Hz | 1 (mono) | Little-endian float32 |

### 4.3 Protocol Flow

```
Client                                    Server
  │◄──── transcription_session.created ─────│
  │──── transcription_session.update ──────►│
  │◄──── transcription_session.updated ─────│
  │──── input_audio_buffer.append ─────────►│ (concurrent)
  │◄──── transcription.delta ───────────────│ (concurrent)
  │──── input_audio_buffer.commit ─────────►│
  │◄──── input_audio_buffer.committed ──────│
```

### 4.4 Event Reference

#### Client Events (you send)

| Event | Description |
|-------|-------------|
| `transcription_session.update` | Configure session (audio format, language) |
| `input_audio_buffer.append` | Send audio chunk (base64 encoded) |
| `input_audio_buffer.commit` | Signal end of audio |

#### Server Events (you receive)

| Event | Description |
|-------|-------------|
| `transcription_session.created` | Session initialized |
| `transcription_session.updated` | Configuration confirmed |
| `conversation.item.input_audio_transcription.delta` | Transcription fragment |
| `input_audio_buffer.committed` | All audio processed |

### 4.5 Prepare Sample Audio

In [ ]:
# Generate a simple test audio (3 seconds of silence for testing connectivity)
# In production, replace with actual audio data
SAMPLE_RATE = 24000
DURATION_SECONDS = 3
CHUNK_DURATION_MS = 80  # 80ms chunks recommended

# Generate silent audio for testing (replace with actual audio in production)
num_samples = SAMPLE_RATE * DURATION_SECONDS
audio_data = np.zeros(num_samples, dtype=np.int16)

# Calculate chunk size
chunk_samples = int(SAMPLE_RATE * CHUNK_DURATION_MS / 1000)
num_chunks = len(audio_data) // chunk_samples

print(f"Audio duration: {DURATION_SECONDS} seconds")
print(f"Sample rate: {SAMPLE_RATE} Hz")
print(f"Chunk duration: {CHUNK_DURATION_MS} ms")
print(f"Number of chunks: {num_chunks}")

### 4.6 Streaming Inference

In [ ]:
async def transcribe_audio(endpoint_name: str, audio_data: np.ndarray, language: str = "ja"):
    """
    Transcribe audio using Kotoba STT bidirectional streaming.
    
    Args:
        endpoint_name: SageMaker endpoint name
        audio_data: Audio samples as int16 numpy array
        language: Target language ("ja" or "en")
    
    Returns:
        Full transcription text
    """
    # Get AWS credentials
    session = boto3.Session()
    credentials = session.get_credentials().get_frozen_credentials()
    
    # Configure HTTP/2 client
    config = Config(
        endpoint_uri=f"https://runtime.sagemaker.{region}.amazonaws.com:8443",
        region=region,
        aws_access_key_id=credentials.access_key,
        aws_secret_access_key=credentials.secret_key,
        aws_session_token=credentials.token,
        aws_credentials_identity_resolver=StaticCredentialsResolver(),
        auth_scheme_resolver=HTTPAuthSchemeResolver(),
        auth_schemes={"aws.auth#sigv4": SigV4AuthScheme(service="sagemaker")},
    )
    
    client = SageMakerRuntimeHTTP2Client(config=config)
    
    # Start bidirectional stream
    stream = await client.invoke_endpoint_with_bidirectional_stream(
        InvokeEndpointWithBidirectionalStreamInput(
            endpoint_name=endpoint_name,
            model_invocation_path="invocations-bidirectional-stream",
        )
    )
    
    # Helper to send JSON event
    async def send_event(payload: dict):
        body = json.dumps(payload).encode("utf-8")
        request = RequestStreamEventPayloadPart(
            value=RequestPayloadPart(bytes_=body)
        )
        await stream.input_stream.send(request)
    
    # Get output stream
    output = await stream.await_output()
    output_stream = output[1] if isinstance(output, tuple) else output
    
    # Wait for session.created
    message = await output_stream.receive()
    data = json.loads(message.value.bytes_.decode())
    assert data["type"] == "transcription_session.created"
    print("Session created")
    
    # Send session configuration
    await send_event({
        "type": "transcription_session.update",
        "session": {
            "input_audio_format": "pcm16",
            "input_audio_sample_rate": SAMPLE_RATE,
            "input_audio_number_of_channels": 1,
            "turn_detection": False,
            "input_audio_transcription": {
                "language": language,
                "target_language": language,
            }
        }
    })
    
    # Wait for session.updated
    message = await output_stream.receive()
    data = json.loads(message.value.bytes_.decode())
    assert data["type"] == "transcription_session.updated"
    print("Session configured")
    
    # Collect transcription deltas
    transcription_parts = []
    
    # Send audio chunks and receive deltas concurrently
    chunk_samples = int(SAMPLE_RATE * CHUNK_DURATION_MS / 1000)
    
    async def send_audio():
        for i in range(0, len(audio_data), chunk_samples):
            chunk = audio_data[i:i+chunk_samples]
            await send_event({
                "event_id": f"chunk_{i // chunk_samples:07d}",
                "type": "input_audio_buffer.append",
                "audio": base64.b64encode(chunk.tobytes()).decode(),
            })
            await asyncio.sleep(CHUNK_DURATION_MS / 1000)  # Simulate real-time
        
        # Commit buffer
        await send_event({"type": "input_audio_buffer.commit"})
    
    async def receive_deltas():
        while True:
            message = await output_stream.receive()
            if message is None:
                break
            
            data = json.loads(message.value.bytes_.decode())
            event_type = data["type"]
            
            if event_type == "conversation.item.input_audio_transcription.delta":
                delta = data.get("delta", "")
                transcription_parts.append(delta)
                print(f"Delta: {delta}", end="", flush=True)
            elif event_type == "input_audio_buffer.committed":
                print("\nBuffer committed")
                break
            elif event_type == "error":
                raise RuntimeError(f"Server error: {data}")
    
    # Run sender and receiver concurrently
    await asyncio.gather(send_audio(), receive_deltas())
    
    # Close stream
    await stream.input_stream.close()
    
    return "".join(transcription_parts)

In [ ]:
# Run transcription
print("Starting transcription...\n")

transcription = await transcribe_audio(
    endpoint_name=ENDPOINT_NAME,
    audio_data=audio_data,
    language="ja",
)

print(f"\n\nFull transcription: {transcription}")

## 5. Clean-up

**Important**: Delete the endpoint when done to avoid ongoing charges.

GPU instances (`ml.g6.2xlarge`) are billed per hour while the endpoint is active.

In [ ]:
# Delete endpoint
print(f"Deleting endpoint '{ENDPOINT_NAME}'...")

sagemaker_session.delete_endpoint(ENDPOINT_NAME)
sagemaker_session.delete_endpoint_config(ENDPOINT_NAME)

print("Endpoint deleted successfully.")

### Unsubscribe from Model Package (Optional)

To unsubscribe from the Kotoba STT model package:

1. Go to [AWS Marketplace Subscriptions](https://console.aws.amazon.com/marketplace/home#/subscriptions)
2. Find "Kotoba STT" in your subscriptions
3. Click **Manage** → **Actions** → **Cancel subscription**

**Note**: You must delete all endpoints using this model package before unsubscribing.

---

## Support

For technical support, please contact Kotoba AI support.

## References

- [SageMaker Bidirectional Streaming Documentation](https://docs.aws.amazon.com/sagemaker/latest/dg/realtime-endpoints-test-endpoints.html)
- [AWS SDK for Python (Boto3)](https://boto3.amazonaws.com/v1/documentation/api/latest/index.html)